# 162 — Representation Similarity

Compare internal representations across layers and positions using CKA,
cosine similarity, and drift metrics to understand how information
transforms through the network.

## Why This Matters

Representation similarity characterizes the structure of internal model representations. Understanding how models organize information — which concepts are linearly separable, how representations cluster, and how they change across layers — is central to interpretability.

**Key references:**
- [Belinkov (2022) "Probing Classifiers"](https://arxiv.org/abs/2102.12452) — Methodology for probing neural representations
- [Kornblith et al. (2019) "Similarity of Neural Network Representations"](https://arxiv.org/abs/1905.00414) — CKA for comparing representations across models and layers

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.representation_similarity import (
    layer_representation_similarity,
    position_representation_similarity,
    component_output_similarity,
    representation_drift,
    cross_input_similarity,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = layer_representation_similarity(model, tokens)
print(f"Mean CKA: {result['mean_similarity']:.4f}  Block-diagonal: {result['block_diagonal_score']:.4f}\n")
print('CKA matrix:')
for i, name_i in enumerate(result['stage_names']):
    row = '  '.join(f'{float(result["similarity_matrix"][i,j]):.3f}' for j in range(len(result['stage_names'])))
    print(f"  {name_i:8s}: {row}")

In [ ]:
result = position_representation_similarity(model, tokens)
for p in result['per_layer']:
    print(f"Layer {p['layer']}: mean_sim={p['mean_similarity']:.4f}  "
          f"most_similar={p['most_similar_pair']}")

In [ ]:
result = component_output_similarity(model, tokens)
for p in result['per_layer']:
    print(f"Layer {p['layer']}: attn⟂mlp cos={p['attn_mlp_cosine']:.4f}  "
          f"attn_norm={p['attn_norm']:.3f}  mlp_norm={p['mlp_norm']:.3f}")

In [ ]:
result = representation_drift(model, tokens)
print(f"Total drift: {result['total_drift']:.4f}\n")
for p in result['per_layer']:
    bar = '#' * int(p['relative_change'] * 30)
    print(f"Layer {p['layer']}: cos={p['cosine_with_previous']:.4f}  "
          f"delta={p['relative_change']:.4f}  {bar}")

In [ ]:
tokens2 = jnp.arange(15, 30)
result = cross_input_similarity(model, tokens, tokens2)
for p in result['per_layer']:
    print(f"Layer {p['layer']}: mean_cos={p['mean_cosine']:.4f}  "
          f"per_pos={[f'{c:.3f}' for c in p['per_position'][:5]]}")